In [1]:
import random

traffic = {
    "North": 40,
    "South": 30,
    "East": 60,
    "West": 20
}

POPULATION_SIZE = 10
GENERATIONS = 50
MIN_GREEN = 10
MAX_GREEN = 60
TOTAL_GREEN_TIME = 120


def create_solution():
    timings = [random.randint(MIN_GREEN, MAX_GREEN)
               for _ in range(4)]

    total = sum(timings)
    timings = [int(t * TOTAL_GREEN_TIME / total)
               for t in timings]

    timings[-1] += TOTAL_GREEN_TIME - sum(timings)

    return timings


def calculate_waiting_time(solution):
    total_waiting = 0

    for green_time, vehicles in zip(solution, traffic.values()):
        waiting = vehicles / green_time
        total_waiting += waiting

    return total_waiting


def fitness(solution):
    return 1 / (1 + calculate_waiting_time(solution))


def select_parent(population):
    candidates = random.sample(population, 3)
    return max(candidates, key=fitness)


def crossover(parent1, parent2):
    point = random.randint(1, 3)

    child = parent1[:point] + parent2[point:]

    total = sum(child)
    child = [int(t * TOTAL_GREEN_TIME / total) for t in child]
    child[-1] += TOTAL_GREEN_TIME - sum(child)

    return child


def mutate(solution):
    if random.random() < 0.2:
        index = random.randint(0, 3)
        change = random.randint(-5, 5)

        solution[index] += change

        solution[index] = max(
            MIN_GREEN, min(MAX_GREEN, solution[index])
        )

        others = [i for i in range(4) if i != index]
        remaining = TOTAL_GREEN_TIME - solution[index]

        if remaining >= MIN_GREEN * 3:
            weights = [solution[i] for i in others]
            weight_sum = sum(weights)

            for i, w in zip(others, weights):
                solution[i] = int(remaining * w / weight_sum)

            solution[others[-1]] += (
                TOTAL_GREEN_TIME - sum(solution)
            )

    return solution


def genetic_algorithm():
    population = [create_solution()
                  for _ in range(POPULATION_SIZE)]

    for generation in range(GENERATIONS):
        new_population = []

        best = min(population, key=calculate_waiting_time)
        new_population.append(best[:])

        while len(new_population) < POPULATION_SIZE:
            parent1 = select_parent(population)
            parent2 = select_parent(population)

            child = crossover(parent1, parent2)
            child = mutate(child)

            new_population.append(child)

        population = new_population

    best_solution = min(population, key=calculate_waiting_time)

    return best_solution


best_timings = genetic_algorithm()

print("Optimized Traffic Signal Timings")
print("--------------------------------")

for road, timing in zip(traffic.keys(), best_timings):
    print(f"{road}: {timing} seconds")

print("\nTotal Green Time:", sum(best_timings), "seconds")
print("Estimated Waiting Score:",
      round(calculate_waiting_time(best_timings), 3))

Optimized Traffic Signal Timings
--------------------------------
North: 31 seconds
South: 27 seconds
East: 39 seconds
West: 23 seconds

Total Green Time: 120 seconds
Estimated Waiting Score: 4.809
